# LIFE on Google Colab — PolitiFact++ (binary MF-vs-MR, LLaMA2-7B)

Faithful reproduction of the paper's setup: binary fake/real over the **LLM pair** (MF=fake, MR=real), with **LLaMA2-7B** as the reconstruction model. End-to-end: **convert → key-sentence extraction → concatenate → features → train**.

**Before you start:**
1. Set the Colab runtime to **GPU** — an **A100** is needed for LLaMA2-7B (Runtime → Change runtime type).
2. Upload the whole `LIFE` repo (including `dataset/data/`) to your Google Drive, e.g. `MyDrive/LIFE`.
3. Edit `PROJECT_DIR` in the path cell below if you put it somewhere else.
4. LLaMA-2 is **gated**: accept the license at https://huggingface.co/meta-llama/Llama-2-7b-hf and have an HF access token ready (login cell below).

Scope: **PolitiFact++** only (~229 LLM-pair articles: 97 fake + 132 real). VLPFN is excluded (its text has no punctuation, so sentence splitting cannot work). GossipCop++ is far heavier; try it only after this works.

In [25]:
# Confirm a GPU is attached
!nvidia-smi

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Sat May 30 02:34:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                       

In [26]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
import os

# <-- change this if you uploaded the repo elsewhere
PROJECT_DIR = '/content/drive/MyDrive/LIFE'
os.chdir(PROJECT_DIR)

POLITIFACT_DIR = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset/PolitiFact++'

# Paper's binary task: LLM pair only (MF=fake, MR=real), reconstructed with LLaMA2-7B.
OUTPUT_BIN     = f'{PROJECT_DIR}/dataset/output_bin'        # MF_fake.jsonl + MR_true.jsonl
KEY_SENT       = f'{PROJECT_DIR}/dataset/keySentence/important_sentences_top10.jsonl'
BERT_CKPT      = f'{PROJECT_DIR}/dataset/bert_bin.pt'       # fresh extractor for MF-vs-MR
FEATURES_LLAMA = f'{PROJECT_DIR}/dataset/features_llama'
TRAIN_PATH     = f'{PROJECT_DIR}/dataset/train_bin.jsonl'
TEST_PATH      = f'{PROJECT_DIR}/dataset/test_bin.jsonl'

print('cwd:', os.getcwd())
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))

cwd: /content/drive/MyDrive/LIFE
PolitiFact++ found: True


In [28]:
# Install dependencies.
# If the fastNLP import fails at the training step, pin a compatible version:
#   !pip install -q fastNLP==1.0.1
!pip install -q -r requirements.txt

In [29]:
# NLTK sentence tokenizer data. 'punkt' gives english.pickle (used by train.py);
# 'punkt_tab' is required by newer nltk's sent_tokenize (used by step 1).
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## HuggingFace login (for gated LLaMA-2)
LLaMA2-7B requires accepting Meta's license on its model page and authenticating with an HF token.

In [30]:
# LLaMA-2 is gated on HuggingFace. First accept the license at
# https://huggingface.co/meta-llama/Llama-2-7b-hf, then run this cell and paste an
# access token from https://huggingface.co/settings/tokens
# (or replace with: login(token="hf_xxx")).
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Step 0 — Convert PolitiFact++ to the binary LLM-pair JSONL
`--subset llm` emits only `MF_fake.jsonl` (97, fake) and `MR_true.jsonl` (132, real) — the paper's binary task. HF/HR (human-written) are not used.

In [31]:
!python dataset/0_convert.py --input_dir "{POLITIFACT_DIR}" --output_dir "{OUTPUT_BIN}" --subset llm

MF.json -> MF_fake.jsonl: 97 records (label=gpt3.5_fake)
MR.json -> MR_true.jsonl: 132 records (label=gpt3.5_true)


## Step 1 — Key-sentence extraction (top-10)
Trains a **fresh** BERT fake/real classifier on MF-vs-MR (saved to `BERT_CKPT`), then keeps the **top-10** most impactful sentences per article (paper's k for PolitiFact++). This is the slowest step (a forward pass per sentence per article).

In [32]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_BIN}" --output_file "{KEY_SENT}" --top_k 10 --model_path "{BERT_CKPT}" --gpu 0

Loading weights: 100% 199/199 [00:00<00:00, 886.67it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing

## Step 2 — Concatenate key sentences back into the data
Adds a `sentence` field to each record in `OUTPUT_BIN` by matching on `(id, label)`. **Overwrites the files in `OUTPUT_BIN` in place** — re-run Step 0 first if you need to reset them.

In [33]:
!python dataset/2_concate.py --folder_path "{OUTPUT_BIN}" --important_sentences_file "{KEY_SENT}"

所有 .jsonl 文件已成功更新。


## Step 3 — Reconstruction probabilities with LLaMA2-7B
The paper's reconstruction model. A malicious prompt is prepended and LLaMA2-7B's per-token log-likelihoods over the key fragments form the "linguistic fingerprint" features. Loads in bfloat16 (~14 GB; needs the A100) and downloads ~13 GB on first run. Writes one feature JSONL per input file into `FEATURES_LLAMA`.

In [36]:
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_BIN}" --output_dir "{FEATURES_LLAMA}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

device: cuda | model: NousResearch/Llama-2-7b-hf | dtype: bfloat16 | scorer: llama
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [00:02<00:00, 120.41it/s, Materializing param=model.norm.weight]
input file:/content/drive/MyDrive/LIFE/dataset/output_bin/MF_fake.jsonl, length:97
  0% 0/97 [00:00<?, ?it/s]0 57
58 122
124 202
206 280
282 306
310 350
352 436
438 484
486 538
542 578
580 648
  1% 1/97 [00:00<00:42,  2.24it/s]0 57
58 122
124 158
161 197
199 247
250 294
296 328
331 421
0 57
58 96
98 128
131 173
175 211
214 276
279 309
311 353
398 422
425 473
476 512
  3% 3/97 [00:00<00:15,  6.11it/s]0 57
58 132
134 208
210 258
260 324
326 406
0 57
58 108
110 160
162 198
201 235
237 275
278 312
314 358
403 441
444 478
480 518
0 57
58 100
102 140
143 171
174 206
208 242
246 276
278 314
318 362
364 386
  6% 6/97 [00:00<00:08, 10.70it/s]0 57
58 118
120 144
146 150
154 212
214 266
270 314
316 350
358 406
408 462
466 524
0 57
58 112
114 144
147 203
205 259
262 314
316

## Step 4 — Train the classifier (binary)
Splits `FEATURES_LLAMA` into train/test and trains the Transformer+CRF classifier for **50 epochs** on the binary MF-vs-MR task. Paper target for PolitiFact++: **Acc 0.900 / F1 0.882**.

In [37]:
!python LIFE_train/train.py \
  --split_dataset \
  --data_path "{FEATURES_LLAMA}" \
  --train_path "{TRAIN_PATH}" \
  --test_path "{TEST_PATH}" \
  --model Transformer \
  --num_train_epochs 50

Log INFO: split dataset...
********************************
The overall data sources:
['MF_fake.jsonl', 'MR_true.jsonl']
100% 183/183 [00:00<00:00, 3622.18it/s]
100% 46/46 [00:00<00:00, 3151.60it/s]

The number of train dataset: 183
The number of test  dataset: 46
********************************
100% 183/183 [00:00<00:00, 836.83it/s]
100% 46/46 [00:00<00:00, 8514.85it/s]
--------------------------------classify--------------------------------
Log INFO: do train...
Epoch:   0% 0/50 [00:00<?, ?it/s]
Iteration:   0% 0/6 [00:00<?, ?it/s]
Iteration:  17% 1/6 [00:00<00:02,  1.72it/s]
Iteration:  33% 2/6 [00:00<00:01,  3.18it/s]
Iteration:  50% 3/6 [00:00<00:00,  4.38it/s]
Iteration:  67% 4/6 [00:00<00:00,  5.38it/s]
Iteration: 100% 6/6 [00:01<00:00,  5.13it/s]
epoch 1: train_loss 2.7119860649108887

Iteration:   0% 0/2 [00:00<?, ?it/s]
Iteration:  50% 1/2 [00:00<00:00,  1.97it/s]
Iteration: 100% 2/2 [00:00<00:00,  2.34it/s]
******** Evalation ********
Accuracy: 51.7
Macro F1 Score: 45.0
Pre

## Notes / troubleshooting
- **HF gating**: LLaMA-2 needs license acceptance + an HF token (see the login cell). A 401/"gated repo" error at Step 3 means that's missing.
- **fastNLP**: if Step 4 errors on `from fastNLP.modules.torch import ...`, run `!pip install -q fastNLP==1.0.1` and restart the runtime.
- **Checkpoints**: `BERT_CKPT` (step 1) and `linear_en.pt` (step 4) are written under `PROJECT_DIR` on Drive, so they survive disconnects.
- **NaN features**: if Step 3 prints NaN/inf, switch Step 3 to `--dtype float32` (fits the 40 GB A100).
- **Re-runs**: Step 2 mutates `OUTPUT_BIN` in place; always re-run Step 0 before re-running Steps 1–3 from scratch.